## Adaptive ORB Candidate Selection

### Overview 

Can daily OHLCV + instrument metadata improve the selection of stocks likely to 
produce profitable ORB trades the next day?

Core contribution:

Daily price/volume data
+ sector / industry metadata
→ candidate ranking model
→ ORB backtest
→ compare against naive selection

Baselines to compare:

1. Top volume stocks
2. Top ATR stocks
3. Random sector-balanced selection
4. Your adaptive ranking model

Good target label:

profitable_orb_next_day = 1 if ORB trade on next day has positive return

Useful features:

- 1d / 5d / 20d return
- 5d / 20d volatility
- ATR
- volume spike ratio
- gap %
- close position in daily range
- sector
- industry
- country

Paper-style title:

Adaptive Candidate Selection for Event-Driven Intraday Breakout Systems Using Daily Market Features

### Motivation 

My current work on ORB and MCI based trading systems 

### data selection caveat 

using current list of snp500 stocks. might result in potential survivorship bias. 



In [2]:
import pandas as pd 
import talib as ta
import pandas_ta as pta
import yfinance as yf 
import numpy as np

In [2]:
snp500_instruments = pd.read_csv("/Users/pankajti/dev/git/event-driven-narrative-trading/icaif_2026/data/snp_500_instruments.csv")

In [3]:
snp500_instruments = snp500_instruments[['symbol','sector', 'industry', 'sub_industry']]

In [4]:
yf_tickers  = yf.Tickers(list(snp500_instruments.symbol))
sp_history = yf_tickers.history(start = "2025-01-01",end="2026-06-12")

[**********************74%***********            ]  381 of 516 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BRK B"}}}
$BRK B: possibly delisted; no timezone found
[**********************80%*************          ]  411 of 516 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BF B"}}}
$BF B: possibly delisted; no timezone found
[*********************100%***********************]  516 of 516 completed

2 Failed downloads:
['BRK B', 'BF B']: possibly delisted; no timezone found


In [8]:
sp_history

Price      Adj Close             Close                                      \
Ticker          BF B BRK B           A        AAPL        ABBV        ABNB   
Date                                                                         
2025-01-02       NaN   NaN  132.063110  242.301926  170.578766  131.479996   
2025-01-03       NaN   NaN  134.299988  241.815033  172.270859  135.710007   
2025-01-06       NaN   NaN  135.032394  243.444626  171.206177  135.199997   
2025-01-07       NaN   NaN  136.002365  240.672333  170.664307  131.289993   
2025-01-08       NaN   NaN  135.596573  241.159210  169.685181  130.800003   
...              ...   ...         ...         ...         ...         ...   
2026-06-05       NaN   NaN  135.440002  307.339996  227.229996  133.539993   
2026-06-08       NaN   NaN  132.690002  301.540009  223.070007  134.429993   
2026-06-09       NaN   NaN  135.479996  290.549988  225.419998  131.350006   
2026-06-10       NaN   NaN  131.619995  291.579987  224.949997  129.100006   
2026-06-11       NaN   NaN  129.550003  295.630005  224.770004  130.869995   

Price                                                      ...   Volume  \
Ticker             ABT       ACGL         ACN        ADBE  ...     WYNN   
Date                                                       ...            
2025-01-02  110.078766  91.379997  341.018005  441.000000  ...  2001600   
2025-01-03  110.457222  91.400002  345.935516  430.570007  ...  2805900   
2025-01-06  109.690628  90.739998  343.471832  431.179993  ...  2923500   
2025-01-07  110.039955  92.250000  348.418701  422.630005  ...  2195300   
2025-01-08  110.864769  92.660004  349.728760  419.579987  ...  1862300   
...                ...        ...         ...         ...  ...      ...   
2026-06-05   91.070000  91.190002  178.250000  251.440002  ...  1380800   
2026-06-08   90.500000  89.610001  174.429993  244.990005  ...  1047300   
2026-06-09   91.250000  90.410004  173.470001  237.880005  ...  2408300   
2026-06-10   89.169998  91.309998  170.500000  233.380005  ...  1438300   
2026-06-11   89.650002  91.129997  167.520004  218.800003  ...  1952700   

Price                                                                        \
Ticker          XEL       XOM      XYL       XYZ      YUM      ZBH     ZBRA   
Date                                                                          
2025-01-02  2817200  12685400  1015900   5695400  2609900   833900   342700   
2025-01-03  3157200  14237900   816600  10633400  1154300  1253700   357900   
2025-01-06  3657200  15623700  1166500   6464800  2120800  1788500   304400   
2025-01-07  3017800  12625900  1232400   6851700  2103000  1649100   353800   
2025-01-08  3714700  17858100  1274500   5932700  2025400  2385600   413600   
...             ...       ...      ...       ...      ...      ...      ...   
2026-06-05  5334900  16584000  1973300   4888100  1782100  1963600  1154900   
2026-06-08  8561500  13870900  1662100   5792700  2426000  1645500  1011300   
2026-06-09  5725800  18163500  2222400   5407800  2579000  1869800   965900   
2026-06-10  7119500  14596500  1695400   3795400  2115200  2393500   918900   
2026-06-11  6943000  17285800  1424600   5934200  2229100  1995900   888000   

Price                         
Ticker           ZS      ZTS  
Date                          
2025-01-02   855100  2232800  
2025-01-03  1051900  2206400  
2025-01-06  1155800  2733900  
2025-01-07  2198600  2488500  
2025-01-08  1686800  2353200  
...             ...      ...  
2026-06-05  4617900  5297000  
2026-06-08  3460500  5432100  
2026-06-09  4774100  6991600  
2026-06-10  3928400  7064400  
2026-06-11  3810600  7262000  

[361 rows x 3610 columns]

In [9]:
history_long

Price,Date,Ticker,open,high,low,close,volume
0,2025-01-02,A,133.824890,134.339552,131.508850,132.063110,953600.0
1,2025-01-02,AAPL,247.349662,247.518596,240.284814,242.301926,55740700.0
2,2025-01-02,ABBV,169.837286,170.901972,169.114807,170.578766,4092000.0
3,2025-01-02,ABNB,131.869995,134.229996,130.410004,131.479996,2605800.0
4,2025-01-02,ABT,110.321358,110.583355,109.418914,110.078766,3569100.0
...,...,...,...,...,...,...,...
184967,2026-06-11,YUM,151.190002,154.350006,150.690002,153.270004,2229100.0
184968,2026-06-11,ZBH,87.510002,88.430000,86.010002,87.139999,1995900.0
184969,2026-06-11,ZBRA,215.970001,222.639999,211.860001,222.440002,888000.0
184970,2026-06-11,ZS,122.000000,126.349998,119.900002,126.110001,3810600.0


In [4]:
import numpy as np
import pandas as pd


def add_features(df):
    """
    df columns:
        date
        symbol
        open
        high
        low
        close
        volume
        sector
        industry
    """

    df = df.sort_values(["symbol", "date"]).copy()

    g = df.groupby("symbol")

    # -----------------------
    # Returns
    # -----------------------

    df["return_1d"] = g["close"].pct_change(1)

    df["return_5d"] = (
        df["close"] /
        g["close"].shift(5)
        - 1
    )

    df["return_20d"] = (
        df["close"] /
        g["close"].shift(20)
        - 1
    )

    # -----------------------
    # Volatility
    # -----------------------

    df["vol_5d"] = (
        g["return_1d"]
        .rolling(5)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["vol_20d"] = (
        g["return_1d"]
        .rolling(20)
        .std()
        .reset_index(level=0, drop=True)
    )

    # -----------------------
    # ATR
    # -----------------------

    prev_close = g["close"].shift(1)

    tr1 = df["high"] - df["low"]

    tr2 = (
        df["high"] - prev_close
    ).abs()

    tr3 = (
        df["low"] - prev_close
    ).abs()

    true_range = pd.concat(
        [tr1, tr2, tr3],
        axis=1
    ).max(axis=1)

    df["atr_14"] = (
        true_range
        .groupby(df["symbol"])
        .rolling(14)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Normalized ATR
    df["natr_14"] = (
        df["atr_14"] /
        df["close"]
    )

    # -----------------------
    # Dollar volume
    # -----------------------

    df["dollar_volume"] = (
        df["close"] *
        df["volume"]
    )

    df["avg_dollar_volume_20"] = (
        g["dollar_volume"]
        .rolling(20)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # -----------------------
    # Volume spike
    # -----------------------

    avg_volume_20 = (
        g["volume"]
        .rolling(20)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["volume_spike"] = (
        df["volume"] /
        avg_volume_20
    )

    # -----------------------
    # Gap %
    # -----------------------

    prev_close = g["close"].shift(1)

    df["gap_pct"] = (
        df["open"] /
        prev_close
        - 1
    )

    # -----------------------
    # Close location value
    # -----------------------

    daily_range = (
        df["high"] -
        df["low"]
    ).replace(0, np.nan)

    df["close_position"] = (
        (df["close"] - df["low"])
        / daily_range
    )

    return df

In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# reshape the yfinance multi-ticker history into a long dataframe
symbol_cols = ["Open", "High", "Low", "Close", "Volume"]
history_long = (
    sp_history.loc[:, symbol_cols]
    .stack(level=1)
    .reset_index()
    .rename(columns={"Ticker": "symbol", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume", "Date": "date"})
)
history_long = history_long.sort_values(["symbol", "date"]).reset_index(drop=True)

# compute daily features for each ticker
grp = history_long.groupby("symbol", group_keys=False)

history_long["return_1d"] = grp["close"].pct_change(1)
history_long["return_5d"] = grp["close"].pct_change(5)
history_long["return_20d"] = grp["close"].pct_change(20)

history_long["vol_5d"] = grp["close"].pct_change().transform(lambda x: x.rolling(5, min_periods=1).std())
history_long["vol_20d"] = grp["close"].pct_change().transform(lambda x: x.rolling(20, min_periods=1).std())

history_long["prev_close"] = grp["close"].shift(1)
history_long["tr1"] = history_long["high"] - history_long["low"]
history_long["tr2"] = (history_long["high"] - history_long["prev_close"]).abs()
history_long["tr3"] = (history_long["low"] - history_long["prev_close"]).abs()
history_long["true_range"] = history_long[["tr1", "tr2", "tr3"]].max(axis=1)
history_long["atr"] = grp["true_range"].transform(lambda x: x.rolling(14, min_periods=1).mean())

history_long["vol_spike"] = history_long["volume"] / grp["volume"].transform(lambda x: x.rolling(20, min_periods=1).mean())
history_long["gap_pct"] = (history_long["open"] - history_long["prev_close"]) / history_long["prev_close"]


/var/folders/tz/k1k21d6x7j1d90h0t6dqf5yc0000gn/T/ipykernel_51709/338165298.py:12: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=1)


In [1]:

history_long["close_pos"] = np.where(
    history_long["high"] != history_long["low"],
    (history_long["close"] - history_long["low"]) / (history_long["high"] - history_long["low"]),
    0.5,
)

history_long["next_open"] = grp["open"].shift(-1)
history_long["next_close"] = grp["close"].shift(-1)
history_long["profitable_orb_next_day"] = (
    (history_long["next_close"] / history_long["next_open"] - 1) > 0
).astype(int)

# merge instrument metadata
orb_features = history_long.merge(
    snp500_instruments,
    on="symbol",
    how="left",
)

# select features and drop rows with missing values
numeric_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "vol_5d",
    "vol_20d",
    "atr",
    "vol_spike",
    "gap_pct",
    "close_pos",
]
categorical_features = ["sector", "industry", "sub_industry"]

orb_features = orb_features.dropna(subset=numeric_features + categorical_features + ["profitable_orb_next_day"])

X = orb_features[numeric_features + categorical_features]
y = orb_features["profitable_orb_next_day"]

preprocessor = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), categorical_features),
    ],
    remainder="passthrough",
)

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=300)),
    ]
)

# train on earlier data and score in a later window
train_mask = orb_features["date"] < "2026-01-01"
test_mask = orb_features["date"] >= "2026-01-01"

model.fit(X[train_mask], y[train_mask])
orb_features["adaptive_score"] = model.predict_proba(X)[:, 1]

print("Adaptive model test AUC:", roc_auc_score(y[test_mask], model.predict_proba(X[test_mask])[:, 1]))

# candidate ranking for a specific date
as_of_date = pd.Timestamp("2026-06-11")
candidate_day = orb_features[orb_features["date"] == as_of_date].copy()

top_volume = candidate_day.nlargest(10, "volume")[["symbol", "sector", "industry", "volume"]]
top_atr = candidate_day.nlargest(10, "atr")[["symbol", "sector", "industry", "atr"]]
adaptive_candidates = candidate_day.nlargest(10, "adaptive_score")[["symbol", "sector", "industry", "adaptive_score"]]
sector_balanced = (
    candidate_day.groupby("sector", group_keys=False)
    .apply(lambda g: g.sample(n=1, random_state=42))
    .reset_index(drop=True)
    .sort_values("sector")
    [["symbol", "sector", "industry", "adaptive_score"]]
)

print("Top volume candidates")
print(top_volume)

print("\nTop ATR candidates")
print(top_atr)

print("\nAdaptive ranking candidates")
print(adaptive_candidates)

print("\nRandom sector-balanced candidates")
print(sector_balanced)

NameError: name 'np' is not defined

In [ ]:
sp_close = sp_history['Close']